In [1]:
import pandas as pd
import numpy as np
import re
from textblob import TextBlob
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
import kaggle
from kagglehub import KaggleDatasetAdapter
from kaggle.api.kaggle_api_extended import KaggleApi

### Load dataset

Dataset initially gathered from multiple sources but due to issues with previous datasets, I decided to use just a combined dataset from Kaggle that has clean location information including Province. The downside is that this just covered 6 metro areas

In [2]:
# #uncomment below to redownload
# # Set the path where you want to save the downloaded file
# download_path = "../data"
# #source = 'jiashenliu/515k-hotel-reviews-data-in-europe'
# source = 'huypui/data-515k-rating-hotel' #with province (city)
# # Download the dataset using kaggle API
# kaggle.api.dataset_download_files(source, path=download_path, unzip=True)

In [3]:
# Now load the CSV file using pandas
file_path = "../data/Hotel_Reviews.csv" 
df = pd.read_csv(file_path)
df = df.sample(100000, random_state=42) # For testing locally
df.head()

,Id_Hotel_Rating,Hotel_Address,Province_Name,Country_Name,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
488440,488441,Via Senigallia 6 20161,Milan,Italy,904,7/21/2017,8.1,Hotel Da Vinci,United Kingdom,Would have appreciated a shop in the hotel th...,52,16670,Hotel was great clean friendly staff free bre...,62,1,9.6,"[' Leisure trip ', ' Couple ', ' Double Room '...",13 days,45.533137,9.171102
274649,274650,Arlandaweg 10 Westpoort 1043 EW,Amsterdam,Netherlands,612,12/12/2016,8.6,Urban Lodge Hotel,Belgium,No tissue paper box was present at the room,10,5018,No Positive,0,7,8.8,"[' Leisure trip ', ' Group ', ' Triple Room ',...",234 day,52.385649,4.834443
374688,374689,Mallorca 251 Eixample 08008,Barcelona,Spain,46,11/26/2015,8.3,Alexandra Barcelona A DoubleTree by Hilton,Sweden,Pillows,3,351,Nice welcoming and service,5,15,7.9,"[' Business trip ', ' Solo traveler ', ' Twin ...",616 day,41.393192,2.161520
404352,404353,Piazza Della Repubblica 17 Central Station 20124,Milan,Italy,241,10/17/2015,9.1,Hotel Principe Di Savoia,United States of America,No Negative,0,1543,Everything including the nice upgrade The Hot...,27,9,10.0,"[' Leisure trip ', ' Couple ', ' Ambassador Ju...",656 day,45.479888,9.196298
451596,451597,Singel 303 309 Amsterdam City Center 1012 WJ,Amsterdam,Netherlands,834,5/16/2016,9.1,Hotel Esther a,United Kingdom,No Negative,0,4687,Lovely hotel v welcoming staff,7,2,9.6,"[' Business trip ', ' Solo traveler ', ' Class...",444 day,52.370545,4.888644


In [4]:
# Check coverage
df['Province_Name'].unique().tolist()

['Milan ', 'Amsterdam ', 'Barcelona ', 'London', 'Paris ', 'Vienna ']

In [5]:
# Province_Name is actually Metro_Area so renaming that column

df.rename(columns={'Province_Name': 'Metro_Area'}, inplace=True)

# Remove Trailing Whitespace
df["Metro_Area"] = df["Metro_Area"].str.strip()
df['Metro_Area'].unique().tolist()

['Milan', 'Amsterdam', 'Barcelona', 'London', 'Paris', 'Vienna']

In [6]:
df.groupby("Metro_Area")[['lat', 'lng']].nunique()

,lat,lng
Metro_Area,,
Amsterdam,105,105
Barcelona,208,208
London,397,397
Milan,160,160
Paris,454,454
Vienna,147,147


### EDA, data cleaning, and feature engineering

In [7]:
# Keep only columns that are needed

df = df[[
    "Hotel_Name",
    "Hotel_Address",
    "Reviewer_Score",
    "Total_Number_of_Reviews",
    "Positive_Review",
    "Negative_Review",
    "lat",
    "lng",
    "Metro_Area"
]].copy()

#df.dropna(subset=["Positive_Review", "Negative_Review", "lat", "lng"], how="all", inplace=True)
df.dropna(subset=["Positive_Review", "Negative_Review"], how="all", inplace=True)
df.head()

,Hotel_Name,Hotel_Address,Reviewer_Score,Total_Number_of_Reviews,Positive_Review,Negative_Review,lat,lng,Metro_Area
488440,Hotel Da Vinci,Via Senigallia 6 20161,9.6,16670,Hotel was great clean friendly staff free bre...,Would have appreciated a shop in the hotel th...,45.533137,9.171102,Milan
274649,Urban Lodge Hotel,Arlandaweg 10 Westpoort 1043 EW,8.8,5018,No Positive,No tissue paper box was present at the room,52.385649,4.834443,Amsterdam
374688,Alexandra Barcelona A DoubleTree by Hilton,Mallorca 251 Eixample 08008,7.9,351,Nice welcoming and service,Pillows,41.393192,2.161520,Barcelona
404352,Hotel Principe Di Savoia,Piazza Della Repubblica 17 Central Station 20124,10.0,1543,Everything including the nice upgrade The Hot...,No Negative,45.479888,9.196298,Milan
451596,Hotel Esther a,Singel 303 309 Amsterdam City Center 1012 WJ,9.6,4687,Lovely hotel v welcoming staff,No Negative,52.370545,4.888644,Amsterdam


***Per data analysis, extracting actual city name from address will not be an efficient clean effort so due to time contraints, I have decided to use Metro_Area as a city. Extracting actual city using lat/long may be useful in a future iteration.***

In [8]:
df["city"] = df["Metro_Area"]
df["city"].value_counts()

city
London       50791
Paris        11675
Barcelona    11673
Amsterdam    10982
Vienna        7585
Milan         7294
Name: count, dtype: int64

In [9]:
sorted(df["city"].dropna().unique())

['Amsterdam', 'Barcelona', 'London', 'Milan', 'Paris', 'Vienna']

### Sentiment analysis

In [ ]:
def polarity(text):
    text = str(text).strip()
    if not text:
        return 0.0
    return TextBlob(text).sentiment.polarity

In [ ]:
df["pos_sentiment"] = df["Positive_Review"].apply(polarity)
df["neg_sentiment"] = df["Negative_Review"].apply(polarity)

In [ ]:
df["pos_sentiment"].describe()

In [ ]:
df["neg_sentiment"].describe()

In [ ]:
df["net_sentiment"] = df["pos_sentiment"] - df["neg_sentiment"]
df["net_sentiment"].describe()

### Quiet vs crowded NLP features

In [ ]:
# QUIET_KEYWORDS = [
#     "quiet", "peaceful", "calm", "serene", "relaxing",
#     "tranquil", "secluded", "sleep well", "good sleep", "not busy"
# ]

# CROWDED_KEYWORDS = [
#     "crowded", "busy", "packed", "touristy", "overcrowded",
#     "noisy", "loud", "queue", "long lines", "too many people", "no sleep"
# ]

# def has_negation(text, keyword):
#     pattern = rf"(not|no|never|hardly)\s+\w*\s*{re.escape(keyword)}"
#     return re.search(pattern, text) is not None

# def score_review(text):
#     text = str(text).lower()

#     quiet = any(q in text for q in QUIET_KEYWORDS)

#     crowded = False
#     for c in CROWDED_KEYWORDS:
#         if c in text and not has_negation(text, c):
#             crowded = True
#             break

#     return int(quiet), int(crowded)

# df[["quiet_flag", "crowded_flag"]] = df["review_text"].apply(
#     lambda x: pd.Series(score_review(x))
# )

# df[["quiet_flag", "crowded_flag"]].mean()

#add more keywords
QUIET_KEYWORDS = [
    "quiet", "peaceful", "calm", "serene", "relaxing",
    "tranquil", "secluded", "sleep well", "good sleep", "not busy"
]

#add more keywords
CROWDED_KEYWORDS = [
    "crowded", "busy", "packed", "touristy", "overcrowded",
    "noisy", "loud", "queue", "long lines", "too many people"
]

def contains_any(text, keywords):
    text = str(text).lower()
    return int(any(k in text for k in keywords))

df["quiet_in_positive"] = df["Positive_Review"].apply(lambda x: contains_any(x, QUIET_KEYWORDS))
df["quiet_in_negative"] = df["Negative_Review"].apply(lambda x: contains_any(x, QUIET_KEYWORDS))

df["crowded_in_positive"] = df["Positive_Review"].apply(lambda x: contains_any(x, CROWDED_KEYWORDS))
df["crowded_in_negative"] = df["Negative_Review"].apply(lambda x: contains_any(x, CROWDED_KEYWORDS))

In [ ]:
# Quiet praised is good; quiet complained about is bad
df["quiet_net"] = df["quiet_in_positive"] - df["quiet_in_negative"]

# Crowding praised is ambiguous but can indicate "lively";
# crowding complained about is more clearly bad.
df["crowded_net"] = df["crowded_in_positive"] - df["crowded_in_negative"]

### Aggregate to hotel level

In [ ]:
hotel_stats = df.groupby(
    ["Hotel_Name", "city", "lat", "lng"],
    as_index=False
).agg(
    avg_rating=("Reviewer_Score", "mean"),
    review_count=("Reviewer_Score", "count"),
    dataset_total_reviews=("Total_Number_of_Reviews", "max"),
    pos_sentiment=("pos_sentiment", "mean"),
    neg_sentiment=("neg_sentiment", "mean"),
    net_sentiment=("net_sentiment", "mean"),
    quiet_positive_rate=("quiet_in_positive", "mean"),
    quiet_negative_rate=("quiet_in_negative", "mean"),
    crowded_positive_rate=("crowded_in_positive", "mean"),
    crowded_negative_rate=("crowded_in_negative", "mean"),
    quiet_net=("quiet_net", "mean"),
    crowded_net=("crowded_net", "mean"),
).reset_index()


hotel_stats.head()

### Adding distance from city centers

In [ ]:
# City-center coordinates
# Using fixed city-center reference points
city_centers = pd.DataFrame({
    "city": ["Milan", "Amsterdam", "Barcelona", "London", "Paris", "Vienna"],
    "city_center_lat": [45.4668, 52.3728, 41.3825, 51.5073, 48.8535, 48.2084],
    "city_center_lng": [9.1905, 4.8936, 2.1769, -0.1277, 2.3484, 16.3725],
})

hotel_stats["city"] = hotel_stats["city"].astype(str).str.strip()
city_centers["city"] = city_centers["city"].astype(str).str.strip()

hotel_stats = hotel_stats.merge(city_centers, on="city", how="left")

In [ ]:
# ---------------------------------------------------------
# 2. Haversine distance from city center
# ---------------------------------------------------------
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


hotel_stats["distance_from_center_km"] = haversine_km(
    hotel_stats["lat"],
    hotel_stats["lng"],
    hotel_stats["city_center_lat"],
    hotel_stats["city_center_lng"]
)

# ---------------------------------------------------------
# 3. Relative distance within each city
# ---------------------------------------------------------
hotel_stats["distance_percentile_within_city"] = (
    hotel_stats.groupby("city")["distance_from_center_km"]
    .rank(method="average", pct=True)
)

hotel_stats["distance_from_center_pct"] = (
    100 * hotel_stats["distance_percentile_within_city"]
).round(1)

# ---------------------------------------------------------
# 4. Two distance preference features
# ---------------------------------------------------------

# A. Sweet spot:
# favors hotels that are not ultra-central but also not too far out
hotel_stats["distance_sweet_spot"] = (
    1 - (2 * np.abs(hotel_stats["distance_percentile_within_city"] - 0.5))
)

# B. Centrality:
# favors hotels closer to the center
hotel_stats["centrality_score"] = 1 - hotel_stats["distance_percentile_within_city"]


### Normalize features

In [ ]:
# scaler = MinMaxScaler()

# cols_to_scale = [
#     "avg_rating",
#     "review_count",
#     "dataset_total_reviews",
#     "sentiment_score",
#     "quiet_score",
#     "crowded_score"
# ]

# hotel_stats[cols_to_scale] = scaler.fit_transform(hotel_stats[cols_to_scale])
# hotel_stats[cols_to_scale].describe()

cols_to_scale = [
    "avg_rating",
    "review_count",
    "dataset_total_reviews",
    "pos_sentiment",
    "neg_sentiment",
    "net_sentiment",
    "quiet_positive_rate",
    "quiet_negative_rate",
    "crowded_positive_rate",
    "crowded_negative_rate",
    "quiet_net",
    "crowded_net",
]

scaler = MinMaxScaler()
hotel_stats[cols_to_scale] = scaler.fit_transform(hotel_stats[cols_to_scale])

### Hidden gem scoring
Use high rating + positive sentiment + quietness, while penalizing crowding and extreme popularity.

In [ ]:
# ---------------------------------------------------------
# 5. Base hidden-gem score
# IMPORTANT: no distance here
# ---------------------------------------------------------
hotel_stats["hidden_score_base"] = (
    0.30 * hotel_stats["avg_rating"]
    + 0.20 * hotel_stats["net_sentiment"]
    + 0.20 * hotel_stats["quiet_net"]
    - 0.15 * hotel_stats["crowded_negative_rate"]
    - 0.10 * hotel_stats["dataset_total_reviews"]
)


### Rank hotels relative to other hotels within the same city

In [ ]:
# #hotel_stats["city_hidden_percentile"] = hotel_stats.groupby("city")["hidden_score"].rank(method="average", pct=True)
# hotel_stats["city_hidden_percentile"] = (
#     hotel_stats.groupby("city")["hidden_score"]
#     .rank(method="average", pct=True)
# )
# hotel_stats[["Hotel_Name", "city", "hidden_score", "city_hidden_percentile"]].head()

hotel_stats["city_hidden_percentile"] = (
    hotel_stats.groupby("city")["hidden_score_base"]
    .rank(method="average", pct=True)
)

hotel_stats[["Hotel_Name", "city", "hidden_score_base", "city_hidden_percentile"]].head()

### Recommendation function
**If you are already going to a major city, where should you stay to avoid tourist-trap or overcrowded options?**

In [ ]:
def recommend(
    df,
    city=None,
    prefer_quiet=True,
    avoid_crowds=True,
    distance_mode="balanced",   # "central", "balanced", "outer"
    distance_weight=0.05,
    top_k=10
):
    out = df.copy()

    if city is not None:
        out = out[
            out["city"].astype(str).str.strip().str.lower() == city.strip().lower()
        ].copy()

    score = out["hidden_score_base"].copy()

    if prefer_quiet:
        score += 0.15 * out["quiet_net"]

    if avoid_crowds:
        score -= 0.15 * out["crowded_negative_rate"]

    if distance_mode == "central":
        score += distance_weight * out["centrality_score"]

    elif distance_mode == "balanced":
        score += distance_weight * out["distance_sweet_spot"]

    elif distance_mode == "outer":
        score += distance_weight * out["distance_percentile_within_city"]

    out["final_score"] = score

    return out.sort_values("final_score", ascending=False).head(top_k)


### Add explainability

In [ ]:
def explain(row, distance_mode="balanced"):
    reasons = []

    if row["quiet_positive_rate"] > 0.60:
        reasons.append("guests often praise how quiet it is")

    if row["quiet_negative_rate"] < 0.20:
        reasons.append("quietness is rarely mentioned as a downside")

    if row["crowded_negative_rate"] < 0.25:
        reasons.append("fewer complaints about crowds and noise")

    if row["avg_rating"] > 0.70:
        reasons.append("strong average review scores")

    if distance_mode == "central" and row["centrality_score"] > 0.75:
        reasons.append("close to the city center")

    elif distance_mode == "balanced" and row["distance_sweet_spot"] > 0.80:
        reasons.append("well positioned away from the busiest core without being too far out")

    elif distance_mode == "outer" and row["distance_percentile_within_city"] > 0.75:
        reasons.append("farther from the busiest center areas")

    if row["city_hidden_percentile"] > 0.85:
        reasons.append("stands out as an overlooked option within its city")

    if not reasons:
        reasons.append("balanced overall profile")

    return ", ".join(reasons)


### Test

In [ ]:
# Central
# paris_results = recommend(
#     hotel_stats,
#     city="Paris",
#     prefer_quiet=True,
#     avoid_crowds=True,
#     distance_mode="central",
#     distance_weight=0.10,
#     top_k=10
# ).copy()

# Balanced
paris_results = recommend(
    hotel_stats,
    city="Paris",
    prefer_quiet=True,
    avoid_crowds=True,
    distance_mode="balanced",
    distance_weight=0.05,
    top_k=10
)

# Outer
# paris_results = recommend(
#     hotel_stats,
#     city="Paris",
#     prefer_quiet=True,
#     avoid_crowds=True,
#     distance_mode="outer",
#     distance_weight=0.10,
#     top_k=10
# )


paris_results["explanation"] = paris_results.apply(explain, axis=1)

paris_results[[
    "Hotel_Name", "city", "final_score", "city_hidden_percentile", "hidden_score_base", "explanation"
]]

### Interactive map for one city

In [ ]:
city_to_plot = "Paris"
map_df = recommend(hotel_stats, city=city_to_plot, prefer_quiet=True, distance_mode="outer",
                   distance_weight=0.10, top_k=50).copy()

fig = px.scatter_mapbox(
    map_df,
    lat="lat",
    lon="lng",
    hover_name="Hotel_Name",
    hover_data={
        "city": True,
        "avg_rating": ':.3f',
        "review_count": ':.3f',
        "hidden_score_base": ':.3f',
        "city_hidden_percentile": ':.3f',
        "lat": False,
        "lng": False
    },
    color="hidden_score_base",
    size="city_hidden_percentile",
    color_continuous_scale="Viridis",
    zoom=10,
    height=700,
    title=f"Hidden Gems Within {city_to_plot}"
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()

## Compare top results across all covered cities

In [ ]:
top_per_city = (
    hotel_stats.sort_values("hidden_score_base", ascending=False)
    .groupby("city", as_index=False)
    .head(5)
    .copy()
)

top_per_city["explanation"] = top_per_city.apply(explain, axis=1)

top_per_city[[
    "Hotel_Name", "city", "hidden_score_base", "city_hidden_percentile", "explanation"
]].sort_values(["city", "hidden_score_base"], ascending=[True, False])

## Notes for next step
Good next upgrades:
- Explore more LLM manipulation
- Extract actual city from lat/lng
- Add airbnb dataset
- Integrate with hotel pricing dataset for budget considerations
- Improve streamlit app